### Structured Output

You can ask models to return responses in a predefined format or structure. This makes the output easier to read, validate, and use in applications or later processing steps. LangChain provides different ways to define schemas and generate structured responses reliably.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
# os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002285DC61BD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002285DC625D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [10]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [11]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002285DC61BD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002285DC625D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out o

In [12]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. First, it\'s a 2010 science fiction action film directed by Christopher Nolan. The cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, and others. The story is about entering dreams to implant or extract ideas. But I need to be more specific.\n\nThe main character, Dom Cobb, is a thief who enters people\'s subconscious to steal secrets. He\'s offered a chance to erase his criminal past by performing the reverse: planting an idea instead of stealing. This is called "inception." The term is important here. The movie explores the concept of layers of dreams, where each level is deeper and time passes more slowly. There\'s also the idea of a totem, a personal object used to distinguish dreams from reality. Cobb\'s totem is a spinning top.\n\nThe plot involves several characters: Arthur, Ariadne, Eames, and Mal. Mal is Cobb\'s wife, who died by suicide

In [13]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output with parsed structure

model.with_structured_output(Movie, include_raw=True)  
raw ouput + structured output

In [14]:
# include_raw=True
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There\'s a Movie function that requires title, year, director, and rating. I need to fill these in for Inception. The title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb gives it 8.8/10. So I\'ll use that. I need to structure the tool call with these parameters. Make sure all required fields are included. No optional parameters here. Just the four required ones. Let me double-check the function\'s required fields: title, year, director, rating. All set. Now format the JSON correctly.\n', 'tool_calls': [{'id': '28fbhfxyn', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'to

### Nested Structure

In [15]:
# a movie can have 
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor] # nested structure
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD") # float or default is none

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Action', 'Science Fiction', 'Heist'], budget=160.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.
- if in output even if you mentioned str but you got from llm as int then typedict will not show any error
- the output it create is in dict form

In [23]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, "The title of the movie"] # Annotated[actual_type, metadata1, metadata2, metadata3]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie spiderman")
response

{'director': 'Sam Raimi', 'rating': 7.5, 'title': 'Spiderman', 'year': 2002}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [17]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

c:\Users\Administrator\Downloads\b150-genaiops\18_Langchain\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


ImportError: Initializing ChatOpenAI requires the langchain-openai package. Please install it with `pip install langchain-openai`